# Electric Meter Raw Correlation\n\n전기 계량기의 **원본 DB measurement 컬럼**만 기준으로 상관관계를 확인하는 노트북입니다.\n\n- 전처리 결과 사용 안 함\n- 파생변수 사용 안 함\n- weather join 안 함\n- `ems.cr_measurement_1h`에서 해당 계량기의 measurement 목록을 먼저 조회한 뒤 동적으로 피벗\n

In [ ]:
from __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom dotenv import load_dotenv\nfrom IPython.display import display\nfrom sqlalchemy import create_engine, text\n\nPROJECT_ROOT = Path.cwd().resolve().parents[0]\nload_dotenv(PROJECT_ROOT / '.env')\n\nDB_HOST = os.getenv('DB_HOST', 'localhost')\nDB_PORT = os.getenv('DB_PORT', '5432')\nDB_NAME = os.getenv('DB_NAME', 'ems')\nDB_USER = os.getenv('DB_USER', 'postgres')\nDB_PASS = os.getenv('DB_PASS', '')\n\nDATABASE_URL = f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'\nengine = create_engine(DATABASE_URL, pool_pre_ping=True)\n\nSTART_TS = '2018-01-01 00:00:00'\nEND_TS = '2024-12-31 23:59:59'\nTARGET_METER_URN = 'H1.Z16'\n

In [ ]:
MEASUREMENT_SQL = text('''\nSELECT DISTINCT measurement\nFROM ems.cr_measurement_1h\nWHERE meter_urn = :meter_urn\nORDER BY measurement\n''')\n\nwith engine.connect() as conn:\n    measurement_rows = conn.execute(MEASUREMENT_SQL, {'meter_urn': TARGET_METER_URN}).fetchall()\n\nmeasurements = [row[0] for row in measurement_rows]\nprint('TARGET_METER_URN =', TARGET_METER_URN)\nprint('measurement count =', len(measurements))\nprint(measurements)\n

In [ ]:
def build_pivot_sql(measurements: list[str]) -> str:\n    pivot_lines = [\n        f\"MAX(CASE WHEN measurement = '{m}' THEN value END) AS \\\"{m}\\\"\"\n        for m in measurements\n    ]\n    pivot_sql = ',\\n    '.join(pivot_lines)\n    return f'''\nSELECT\n    ts,\n    meter_urn,\n    {pivot_sql}\nFROM ems.cr_measurement_1h\nWHERE meter_urn = :meter_urn\n  AND ts BETWEEN :start_ts AND :end_ts\nGROUP BY ts, meter_urn\nORDER BY ts\n'''\n\npivot_sql = text(build_pivot_sql(measurements))\nraw_df = pd.read_sql(\n    pivot_sql,\n    con=engine,\n    params={\n        'meter_urn': TARGET_METER_URN,\n        'start_ts': START_TS,\n        'end_ts': END_TS,\n    },\n    parse_dates=['ts'],\n)\n\nfor column in measurements:\n    raw_df[column] = pd.to_numeric(raw_df[column], errors='coerce')\n\nprint(raw_df.shape)\ndisplay(raw_df.head())\n

In [ ]:
availability_df = pd.DataFrame({\n    'column': measurements,\n    'non_null_count': [int(raw_df[col].notna().sum()) for col in measurements],\n    'null_ratio': [float(raw_df[col].isna().mean()) for col in measurements],\n})\navailability_df = availability_df.sort_values(['null_ratio', 'column']).reset_index(drop=True)\ndisplay(availability_df)\n

In [ ]:
usable_columns = [col for col in measurements if raw_df[col].notna().sum() >= 2]\ncorr_df = raw_df[usable_columns].corr()\n\nprint('usable_columns =', usable_columns)\ndisplay(corr_df.round(3))\n

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(usable_columns) * 0.7), max(8, len(usable_columns) * 0.7)))\nim = ax.imshow(corr_df, cmap='coolwarm', vmin=-1, vmax=1)\nax.set_xticks(range(len(usable_columns)))\nax.set_yticks(range(len(usable_columns)))\nax.set_xticklabels(usable_columns, rotation=90)\nax.set_yticklabels(usable_columns)\nax.set_title(f'Raw DB Correlation: {TARGET_METER_URN}')\nfig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\nplt.tight_layout()\nplt.show()\n

In [ ]:
pair_rows = []\nfor i, col_a in enumerate(usable_columns):\n    for col_b in usable_columns[i + 1:]:\n        pair_rows.append({\n            'col_a': col_a,\n            'col_b': col_b,\n            'corr': corr_df.loc[col_a, col_b],\n            'abs_corr': abs(corr_df.loc[col_a, col_b]),\n        })\n\npair_corr_df = pd.DataFrame(pair_rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)\ndisplay(pair_corr_df.head(30).round(4))\n

## 사용 방법\n\n1. `TARGET_METER_URN`을 원하는 전기 계량기로 변경\n2. 셀을 순서대로 실행\n3. 원본 DB measurement 기준 null 비율, correlation matrix, 상위 상관쌍 확인\n\n메모:\n- 이 노트북은 전처리 규칙을 적용하지 않습니다.\n- DB에 있는 measurement만 동적으로 피벗합니다.\n- 따라서 계량기마다 컬럼 목록이 다를 수 있습니다.\n